# 00 — EDA: analiza skupa podataka

Kratka analiza `dataset/labels.csv`: koliko imamo slika, kako su rasporedjene klase,
odakle dolaze podaci, i da li je skup balansiran.

**Tim:** Irina Marko, Nikola Lazarević

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
if not (ROOT / 'dataset' / 'labels.csv').exists():
    ROOT = ROOT.parent  # ako se pokrene iz podfoldera

df = pd.read_csv(ROOT / 'dataset' / 'labels.csv')
print('ukupno redova:', len(df))
print('kolone:', list(df.columns))
df.head(3)

## Struktura skupa

Jedan red = jedna slika artikla + labele (category, subcategory, color_family).
Labele su automatske (Item Tree + paleta boja), nisu rucno crtane.

In [ ]:
print('=== po division ===')
print(df['division'].value_counts())
print()
print('=== po source_file (top) ===')
print(df['source_file'].value_counts().head(10))
print()
print('=== match_confidence ===')
print(df['match_confidence'].value_counts())
print()
n_no_color = df['color_family'].isna().sum() + (df['color_family'] == '').sum()
print('bez color_family:', int(n_no_color))

## (Ne)balansiranost klasa

Klase su prilicno neuravnotežene — zato u treningu koristimo class weights i macro-F1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

df['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('category')
axes[0].tick_params(axis='x', rotation=75)

df['subcategory'].value_counts().head(15).plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('subcategory (top 15)')
axes[1].tick_params(axis='x', rotation=75)

df['color_family'].value_counts().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('color_family')
axes[2].tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.show()

print('najcesca category:', df['category'].value_counts().iloc[0],
      '/', 'najredja:', df['category'].value_counts().iloc[-1])

## Train / val / test (posle skripte 04)

Retke klase su izbacene (npr. category min 20 primera). Podela je 70/15/15, stratifikovana.

In [ ]:
report = (ROOT / 'dataset' / 'splits' / 'split_report.txt')
print(report.read_text(encoding='utf-8')[:1800])

## Primeri slika

Nekoliko nasumicnih primera da se vidi kako izgleda ulaz (nekad samo artikal, nekad i model).

In [ ]:
from PIL import Image

sample = df.sample(6, random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = Image.open(ROOT / row['image_path'])
    ax.imshow(img)
    ax.set_title(f"{row['category']}\n{row['subcategory']} / {row['color_family']}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Kratki zakljucci iz EDA

- skup je realan (razliciti brendovi, razlicit kvalitet slika)
- klase su neuravnotežene → macro-F1 + class weights
- deo slika nema boju (marketinska imena) → izbaceno iz color zadatka
- ista silueta u vise boja postoji → kasnije vazno za leakage diskusiju